In [1]:
import sys
sys.executable


'd:\\chatbot\\ankit\\Scripts\\python.exe'

In [2]:
!pip install langchain langchain-community
!pip install faiss-cpu pypdf sentence-transformers
!pip install transformers accelerate torch
!pip install -U langchain langchain-community




[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 3.1 MB/s  0:00:00

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 1.2.5

    Uninstalling langchain-core-1.2.5:

      Successfully uninstalled langchain-core-1.2.5

   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ------ --------------------------------- 1/6 [langchain-core]
   ----


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import langchain
import faiss
import transformers
from langchain_community.document_loaders import PyPDFLoader

print("All imports successful")


All imports successful


In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("D:\chatbot\company_policy.pdf")
documents = loader.load()

print("Pages loaded:", len(documents))


Pages loaded: 1


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))


Total chunks: 2


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_19096\3055314890.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,     
    max_length=256
)


Device set to use cpu


In [10]:
from langchain_community.llms import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=pipe)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_19096\649677049.py:3: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [11]:
query = "What is the leave policy?"

docs = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content)



--- Chunk 1 ---
Company Policy Document
1. Working Hours Policy
Employees are expected to work 9 hours per day, from 9:30 AM to 6:30 PM, Monday to Friday.
Flexible working hours may be approved by the manager based on role and performance.
2. Leave Policy
Employees are entitled to 18 paid leaves per year. This includes casual leave, sick leave, and
earned leave. Leaves must be approved by the reporting manager in advance.
3. Work From Home Policy

--- Chunk 2 ---
3. Work From Home Policy
Work from home is allowed up to 2 days per week with prior approval. Employees must ensure
availability during official working hours.
4. Notice Period Policy
The standard notice period is 60 days for full-time employees. Either party may waive off the notice
period with mutual agreement.
5. Code of Conduct
Employees are expected to maintain professional behavior at all times. Any violation of company
policies may result in disciplinary action.


In [12]:
context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are an assistant answering questions based ONLY on the given context.

Context:
{context}

Question:
{query}

Answer in a clear and concise way.
"""


In [13]:
response = llm.invoke(prompt)
print(response)


Leave Policy Employees are entitled to 18 paid leaves per year. This includes casual leave, sick leave, and earned leave. Leaves must be approved by the reporting manager in advance.


In [14]:
query = "What is the leave policy?"

docs = vectorstore.similarity_search(query, k=3)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are an assistant answering questions based ONLY on the given context.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)
print(response)


Leave Policy Employees are entitled to 18 paid leaves per year. This includes casual leave, sick leave, and earned leave. Leaves must be approved by the reporting manager in advance.
